In [ ]:
#!pip install gensim
#!pip install pprint
#!pip install pyLDAvis
#!pip install nltk
#!pip install --upgrade pip
#!pip install konlpy
#!pip install wordcloud
#!pip install matplotlib
#!pip install networkx

In [ ]:
    import pandas as pd
    import re
    from itertools import combinations
    from konlpy.tag import Kkma
    from konlpy.tag import Okt
    import networkx as nx
    from tqdm import tqdm
    import matplotlib.pyplot as plt
    import re

In [ ]:
# 작업 경로 지정
# data_path =  "C:/Users/legen/Desktop/Lab Project/LG text mining/data/"
# vacuum_script = pd.read_csv(data_path + '무선스틱청소기_pool.csv', encoding='utf-8') # 첫번째 시트 불러오기

# "C:/Users/legen/Desktop/Lab Project/LG text mining/data/무선스틱청소기_pool.csv"

# 의류관리기
styler_script = pd.read_csv("/content/의류관리기_pool.csv", encoding='utf-8')
styler_script['content'] = styler_script['content'].fillna("").astype(str)


# 무선스틱청소기
vacuum_script = pd.read_csv("/content/무선스틱청소기_pool.csv", encoding='utf-8')
vacuum_script['content'] = vacuum_script['content'].fillna("").astype(str)

# 로봇청소기
robot_script = pd.read_csv("/content/로봇청소기_pool.csv", encoding='utf-8')
robot_script['content'] = robot_script['content'].fillna("").astype(str)

## 불용어 제거

In [ ]:
# 불용어 목록
irrelevant_keywords = [
    '구매', '판매', '가격', '비교', '할인', '배송', '무료배송', '최저가', 'Top5'
    'Top 5', 'Top 10', 'Top10', '베스트5', '베스트10', '판매량', '꿀정보', '고민',
    '인기', '쇼핑', '증정', '순위', '가입', '핫딜', '예약', '세일', '원', '만원', 
    '출시', '팝니다', '삽니다', '알뜰', '무상', '어떤가요', '어떤거', '딜', '뉴스',
    '추천해주세요', '괜찮나요', '사드렸어요', '부탁드립니다', '있나요', '미개봉',
    '택포', '부탁드려요', '부탁', '해주세요', '쓰시는분', '인가요', '있나요',
    '있을까요', '괜찮은거', 'top3', '가요?', '당첨자', '혜택', '받아용', '쓰시나요',
    '받아요', '없나요', '좋나요', '살려는데', '시나요', '하나요', '쓰나요', '산다'
]

# 해당 단어가 title에 포함된 데이터를 제거
def remove_irrelevant_titles(data, keywords):

    # 불용어 패턴 생성 (단어를 OR로 묶음)
    pattern = '|'.join(keywords)
    
    # 불용어가 포함된 행 필터링
    filtered_data = data[~data['title'].str.contains(pattern, case=False, na=False)]
    
    return filtered_data

In [ ]:
# styler
# 불용어가 포함된 title 제거
styler_script_filtered = remove_irrelevant_titles(styler_script, irrelevant_keywords)

# 결과 확인
print(f"제거 전 데이터 개수: {len(styler_script)}")
print(f"제거 후 데이터 개수: {len(styler_script_filtered)}")

In [ ]:
# vacuum
# 불용어가 포함된 title 제거
vacuum_script_filtered = remove_irrelevant_titles(vacuum_script, irrelevant_keywords)

# 결과 확인
print(f"제거 전 데이터 개수: {len(vacuum_script)}")
print(f"제거 후 데이터 개수: {len(vacuum_script_filtered)}")

In [ ]:
# robot
# 불용어가 포함된 title 제거
robot_script_filtered = remove_irrelevant_titles(robot_script, irrelevant_keywords)

# 결과 확인
print(f"제거 전 데이터 개수: {len(robot_script)}")
print(f"제거 후 데이터 개수: {len(robot_script_filtered)}")

## 이상한 제목 제거

1. title, content에 링크가 있는 데이터
2. 한글 자음/모음만으로 이루어진 경우
3. 전부 영어와 일부 특수문자만 포함된 경우

In [ ]:
import re

# 삭제 조건을 확인하는 함수 정의
def should_drop(row):
    title = row['title']
    content = row['content']
    
    # 링크 확인 (http/https 포함 여부)
    link_pattern = r'(http|https)://[^\s]+'
    if re.search(link_pattern, title) or re.search(link_pattern, content):
        return True
    
    # 한글 자음/모음만으로 이루어진 경우
    hangul_pattern = r'^[ㄱ-ㅎㅏ-ㅣ\s]+$'
    if re.match(hangul_pattern, title) or re.match(hangul_pattern, content):
        return True
    
    # 전부 영어와 일부 특수문자만 포함된 경우
    english_only_pattern = r'^[a-zA-Z\s.,™-]+$'
    if re.match(english_only_pattern, title) or re.match(english_only_pattern, content):
        return True
    
    # 특정 시작 단어 제거 (예: "How to")
    if title.lower().startswith("how to"):
        return True
    
    return False

In [ ]:
# styler
# 조건에 맞는 행 삭제
styler_script_filtered = styler_script_filtered[~styler_script_filtered.apply(should_drop, axis=1)]

print(styler_script_filtered.shape)

In [ ]:
# vacuum
# 조건에 맞는 행 삭제
vacuum_script_filtered = vacuum_script_filtered[~vacuum_script_filtered.apply(should_drop, axis=1)]

print(vacuum_script_filtered.shape)

In [ ]:
# robot
# 조건에 맞는 행 삭제
robot_script_filtered = robot_script_filtered[~robot_script_filtered.apply(should_drop, axis=1)]

print(robot_script_filtered.shape)

## 필터링 데이터 저장

In [ ]:
# 경로 및 저장 파일명 지정
styler_script_filtered.to_csv('/content/drive/MyDrive/LG/Data/의류관리기_pool_1.csv', index=False, encoding='utf-8-sig')
vacuum_script_filtered.to_csv('/content/drive/MyDrive/LG/Data/무선스틱청소기_pool_1.csv', index=False, encoding='utf-8-sig')
robot_script_filtered.to_csv('/content/drive/MyDrive/LG/Data/로봇청소기_pool_1.csv', index=False, encoding='utf-8-sig')